# Exact GWF samples with JAX replay

This notebook generates small float32 and float64 GWF files with the public PyCBC writer, then compares direct JAX replay with the original CPU `read_frame(check_integrity=True)`. It checks shape, dtype, GPS epoch, sample interval and **every data byte**, including signed zeros, subnormals and quiet NaN payloads. Separate raw and zero-suppressed payload examples explain why decoding introduces no floating-point approximation.

Frames exist only in a temporary directory. PyCBC, JAX, NumPy, pandas and the public LALSuite frame runtime are required. The optional CUDA codec requires the environment described in the [GWF replay guide](../../docs/jax_gwf.rst).

## Setup

Use the first available GPU, or a JAX CPU device. `JAXScheme` enables X64 by default; the explicit check below is needed to preserve float64 words. Printed library versions and hardware describe this execution without exposing machine names or file locations.

In [1]:
from dataclasses import replace
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
import warnings

warnings.filterwarnings("ignore", message="Wswiglal-redir-stdio:")

import jax
import numpy as np
import pandas as pd
from IPython.display import display

from pycbc import scheme
from pycbc.frame import read_frame, write_frame
from pycbc.frame.gwf_jax import compression_info, decode_fr_vect, parse_gwf
from pycbc.frame.gwf_replay_jax import GWFReplaySource
from pycbc.types import TimeSeries

available = jax.devices()
device_choice = "cuda:0" if any(d.platform == "gpu" for d in available) else "cpu"
with scheme.JAXScheme(device_choice) as context:
    assert jax.config.x64_enabled
    hardware = context.jax_device.device_kind
print({"NumPy": np.__version__, "JAX": jax.__version__,
       "JAXlib": version("jaxlib"), "LALSuite": version("lalsuite"),
       "device": device_choice, "hardware": hardware})

{'NumPy': '2.5.2', 'JAX': '0.11.1', 'JAXlib': '0.11.1', 'LALSuite': '7.26.15', 'device': 'cpu', 'hardware': 'cpu'}


## Samples defined by their bits

Constructing values through unsigned integer views preserves the selected NaN payloads and zero signs. Numeric equality alone would miss the zero sign and cannot compare NaNs. Byte comparisons test both.

In [2]:
labels = ["+0", "-0", "smallest positive subnormal",
          "smallest negative subnormal", "+1", "-1", "+infinity", "-infinity",
          "positive quiet NaN payload", "negative quiet NaN payload",
          "largest positive subnormal", "smallest positive normal",
          "+0.25", "-0.25", "+10", "-10"]
patterns = {
    "float32": np.array([
        0x00000000, 0x80000000, 0x00000001, 0x80000001,
        0x3f800000, 0xbf800000, 0x7f800000, 0xff800000,
        0x7fc01234, 0xffc05678, 0x007fffff, 0x00800000,
        0x3e800000, 0xbe800000, 0x41200000, 0xc1200000,
    ], dtype=np.uint32),
    "float64": np.array([
        0x0000000000000000, 0x8000000000000000,
        0x0000000000000001, 0x8000000000000001,
        0x3ff0000000000000, 0xbff0000000000000,
        0x7ff0000000000000, 0xfff0000000000000,
        0x7ff8000000001234, 0xfff8000000005678,
        0x000fffffffffffff, 0x0010000000000000,
        0x3fd0000000000000, 0xbfd0000000000000,
        0x4024000000000000, 0xc024000000000000,
    ], dtype=np.uint64),
}
samples = {dtype: words.view(dtype) for dtype, words in patterns.items()}
display(pd.DataFrame({
    "sample": labels,
    "float32 word": [f"0x{int(word):08x}" for word in patterns["float32"]],
    "float64 word": [f"0x{int(word):016x}" for word in patterns["float64"]],
}))

,sample,float32 word,float64 word
0,+0,0x00000000,0x0000000000000000
1,-0,0x80000000,0x8000000000000000
2,smallest positive subnormal,0x00000001,0x0000000000000001
3,smallest negative subnormal,0x80000001,0x8000000000000001
4,+1,0x3f800000,0x3ff0000000000000
5,-1,0xbf800000,0xbff0000000000000
6,+infinity,0x7f800000,0x7ff0000000000000
7,-infinity,0xff800000,0xfff0000000000000
8,positive quiet NaN payload,0x7fc01234,0x7ff8000000001234
9,negative quiet NaN payload,0xffc05678,0xfff8000000005678


## Public frame writer and original reader

For each dtype, write two contiguous two-second frames at eight samples per second. The second frame reverses the sample order. Compare an entire four-second span, a span crossing the file boundary, and a sample-aligned fractional start. Both the default codec preference and explicit host decompression return JAX-resident samples. On a CPU device the default compressed-vector path also uses host decompression; this run therefore does not establish CUDA codec execution.

In [3]:
epoch = 1_000_000_000
sample_rate = 8
channel = "H1:SYNTHETIC"
spans = [("full", epoch, 4.0),
         ("cross-file", epoch + 1.5, 1.0),
         ("fractional start", epoch + 0.375, 1.0)]
results = []
first_vectors = {}

with TemporaryDirectory() as directory:
    for dtype, values in samples.items():
        frames = []
        for index, block in enumerate((values, values[::-1].copy())):
            start = epoch + 2 * index
            frame = Path(directory) / f"H-SYNTHETIC{values.itemsize * 8}-{start}-2.gwf"
            with scheme.CPUScheme():
                write_frame(str(frame), channel,
                            TimeSeries(block, delta_t=1 / sample_rate, epoch=start))
            container = parse_gwf(frame.read_bytes())
            vector = next(v for v in container.vectors if v.name == channel)
            if index == 0:
                first_vectors[dtype] = vector
            frames.append(str(frame))
        input_span = np.concatenate((values, values[::-1]))
        for label, start, duration in spans:
            with scheme.CPUScheme():
                original = read_frame(frames, channel, start_time=start,
                                      duration=duration, check_integrity=True)
                expected = original.numpy().copy()
                expected_epoch = str(original.start_time)
                expected_delta_t = original.delta_t
            first = int(round((start - epoch) * sample_rate))
            count = int(round(duration * sample_rate))
            assert expected.tobytes() == input_span[first:first + count].tobytes()
            for codec in ("deflate", "host"):
                with scheme.JAXScheme(device_choice):
                    replay = GWFReplaySource(frames, channel, sample_rate,
                                             cuda_codec=codec).read(start, duration)
                    actual = replay.numpy()
                    assert actual.shape == expected.shape
                    assert actual.dtype == expected.dtype
                    assert actual.tobytes() == expected.tobytes()
                    assert str(replay.start_time) == expected_epoch
                    assert replay.delta_t == expected_delta_t
                    assert replay._data.array.device.platform in ("cpu", "gpu")
                results.append({"dtype": dtype, "span": label,
                                "codec preference": codec, "samples": count,
                                "GPS epoch": expected_epoch,
                                "delta_t (s)": expected_delta_t,
                                "data and metadata exact": True})
display(pd.DataFrame(results))

,dtype,span,codec preference,samples,GPS epoch,delta_t (s),data and metadata exact
0,float32,full,deflate,32,1000000000.0,0.125,True
1,float32,full,host,32,1000000000.0,0.125,True
2,float32,cross-file,deflate,8,1000000001.5,0.125,True
3,float32,cross-file,host,8,1000000001.5,0.125,True
4,float32,fractional start,deflate,8,1000000000.375,0.125,True
5,float32,fractional start,host,8,1000000000.375,0.125,True
6,float64,full,deflate,32,1000000000.0,0.125,True
7,float64,full,host,32,1000000000.0,0.125,True
8,float64,cross-file,deflate,8,1000000001.5,0.125,True
9,float64,cross-file,host,8,1000000001.5,0.125,True


## Raw payloads: assemble words, then bitcast

The public writer chooses its own compression. To isolate raw decoding, reuse only a parsed vector's metadata and replace its payload with the specified little-endian sample bytes. This is a payload-level example, not another full-container writer. The decoder combines byte values with shifts and bitwise OR, then bitcasts each unsigned word to float32 or float64. It does not numerically convert an integer to a float.

In [4]:
payload_results = []
for dtype, words in patterns.items():
    vector = first_vectors[dtype]
    raw_code = 256 if vector.version == 8 else 0x8000
    raw_vector = replace(
        vector, compression=compression_info(vector.version, raw_code),
        payload=words.astype(words.dtype.newbyteorder("<")).tobytes())
    with scheme.JAXScheme(device_choice):
        decoded = np.asarray(decode_fr_vect(raw_vector))
    assert decoded.shape == samples[dtype].shape
    assert decoded.dtype == samples[dtype].dtype
    assert decoded.tobytes() == samples[dtype].tobytes()
    payload_results.append({"dtype": dtype, "payload": "raw",
                            "shape and dtype exact": True, "all bytes exact": True})
display(pd.DataFrame(payload_results))

,dtype,payload,shape and dtype exact,all bytes exact
0,float32,raw,True,True
1,float64,raw,True,True


## Zero suppression: modular integer reconstruction

The [version-8 GWF specification, Appendix B](https://dcc.ligo.org/public/0000/T970130/002/T970130-v2.pdf) defines integer differences even for floating-point samples. Each block records its word width and biased differences. This small independent encoder uses one full-width block so arbitrary special-value words fit. It intentionally prioritizes clarity over compression ratio.

Decode subtracts the integer bias, cumulatively adds differences modulo $2^{32}$ or $2^{64}$, then bitcasts the reconstructed words. Modular integer addition is associative, so the parallel prefix scan has exactly the same result as sequential addition. There is no rounded floating-point subtraction or sum.

In [5]:
def one_block_zero_payload(words):
    width = words.dtype.itemsize * 8
    mask = (1 << width) - 1
    bias = (1 << (width - 1)) - 1
    packed = width - 1
    position = width.bit_length() - 1
    previous = 0
    for word in words:
        difference = (int(word) - previous) & mask
        packed |= ((difference + bias) & mask) << position
        previous = int(word)
        position += width
    byte_count = 2 * ((position + 15) // 16)
    return len(words).to_bytes(2, "little") + packed.to_bytes(byte_count, "little")

for dtype, words in patterns.items():
    vector = first_vectors[dtype]
    zero_code = (264 if words.itemsize == 4 else 266) if vector.version == 8 else 0x8001
    zero_vector = replace(
        vector, compression=compression_info(vector.version, zero_code),
        payload=one_block_zero_payload(words))
    with scheme.JAXScheme(device_choice):
        decoded = np.asarray(decode_fr_vect(zero_vector))
    assert decoded.shape == samples[dtype].shape
    assert decoded.dtype == samples[dtype].dtype
    assert decoded.tobytes() == samples[dtype].tobytes()
    payload_results.append({"dtype": dtype, "payload": "zero-suppressed",
                            "shape and dtype exact": True, "all bytes exact": True})
display(pd.DataFrame(payload_results))

,dtype,payload,shape and dtype exact,all bytes exact
0,float32,raw,True,True
1,float64,raw,True,True
2,float32,zero-suppressed,True,True
3,float64,zero-suppressed,True,True


## What these checks establish

The generated frames match the original CPU reader exactly for both real dtypes, all tested spans and both codec preferences. Raw and zero-suppressed examples retain every tested word, including zero signs, subnormals and NaN payloads. They demonstrate lossless decoding, rather than a numerical tolerance.

The synthetic payload examples bypass container checksums by construction; full-frame comparisons use the production parser and the original reader's integrity check. Unsupported topology or codecs require the compatibility reader, while malformed admitted data and supported checksum mismatches raise errors. These examples do not validate every GWF layout, CUDA execution on a CPU-only run, or subsequent filtering and conditioning.